In [1]:
# ===== Summarize extraction-test results: counts & percentages of A/B/C per LLM =====
# Looks for "0-*-ABC-regex_llm.csv" files,
# parses the "answer" column, and reports counts & percents per model.
# Also reports how many rows were extracted by regex vs llm.
#
# Outputs:
#   - out/temp_0_extraction_choice_counts.csv
#   - out/temp_0_extraction_choice_counts_by_model.tex

import os, re, glob
import pandas as pd

BASE_DIR = "."
PATTERN  = "0-*-ABC-regex_llm.csv"
OUT_DIR  = os.path.join(BASE_DIR, "out")
os.makedirs(OUT_DIR, exist_ok=True)

def model_name_from_path(p: str) -> str:
    fname = os.path.basename(p)
    fname = re.sub(r"^0-", "", fname)
    fname = re.sub(r"-ABC-regex_llm\.csv$", "", fname)
    return fname

def extract_choice_letter(ans: str):
    if not isinstance(ans, str) or not ans.strip():
        return None
    s = re.sub(r'^\s*Answer:\s*', '', ans, flags=re.IGNORECASE).strip()
    m = re.match(r'^\s*([ABC])\b', s, flags=re.IGNORECASE)
    if not m:
        m = re.search(r'\b([ABC])\b', s[:120], flags=re.IGNORECASE)
    return m.group(1).upper() if m else None

def load_and_count_choice(path: str) -> dict:
    df = pd.read_csv(path)

    if "answer" not in df.columns:
        raise ValueError(f"{path} is missing an 'answer' column.")

    choices = df["answer"].map(extract_choice_letter)
    valid = choices.dropna()
    counts = valid.value_counts()

    regex_n = 0
    llm_n = 0
    failed_n = 0
    if "extraction_method" in df.columns:
        method_counts = df["extraction_method"].fillna("").value_counts()
        regex_n = int(method_counts.get("regex", 0))
        llm_n = int(method_counts.get("llm", 0))
        failed_n = int(method_counts.get("failed", 0))

    return {
        "A": int(counts.get("A", 0)),
        "B": int(counts.get("B", 0)),
        "C": int(counts.get("C", 0)),
        "total": int(valid.shape[0]),
        "invalid_rows": int(df.shape[0] - valid.shape[0]),
        "regex_rows": regex_n,
        "llm_rows": llm_n,
        "failed_rows": failed_n,
    }

files = sorted(glob.glob(os.path.join(BASE_DIR, PATTERN)))
if not files:
    raise SystemExit(f"No files matched {PATTERN} in {BASE_DIR}")

rows = []
for f in files:
    model = model_name_from_path(f)
    stats = load_and_count_choice(f)
    A, B, C, total = stats["A"], stats["B"], stats["C"], stats["total"]

    rows.append({
        "model": model,
        "A": A, "B": B, "C": C, "total": total,
        "pct_A": round(100.0 * A / total, 2) if total else 0.0,
        "pct_B": round(100.0 * B / total, 2) if total else 0.0,
        "pct_C": round(100.0 * C / total, 2) if total else 0.0,
        "invalid_rows": stats["invalid_rows"],
        "regex_rows": stats["regex_rows"],
        "llm_rows": stats["llm_rows"],
        "failed_rows": stats["failed_rows"],
        "source_file": os.path.basename(f),
    })

summary = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

csv_path = os.path.join(OUT_DIR, "temp_0_extraction_choice_counts.csv")
summary.to_csv(csv_path, index=False)
print(f"[OK] Wrote {csv_path}")

print("\nCounts and percentages by model (temperature = 0, extraction = regex_llm):")
print(summary[
    ["model","A","B","C","total","pct_A","pct_B","pct_C","regex_rows","llm_rows","invalid_rows"]
].to_string(index=False))

try:
    from textwrap import dedent

    latex_cols = [
        "model","A","B","C","total","pct_A","pct_B","pct_C","regex_rows","llm_rows","invalid_rows"
    ]
    latex_df = summary[latex_cols].copy()
    latex_df = latex_df.rename(columns={
        "pct_A": r"\makecell{\%\\A}",
        "pct_B": r"\makecell{\%\\B}",
        "pct_C": r"\makecell{\%\\C}",
        "regex_rows": r"\makecell{regex\\rows}",
        "llm_rows": r"\makecell{llm\\rows}",
        "invalid_rows": r"\makecell{invalid\\rows}"
    })

    tex_path = os.path.join(OUT_DIR, "temp_0_extraction_choice_counts_by_model.tex")
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(dedent(r"""
        % Requires: \usepackage{booktabs}
        % Optional for multi-line headers: \usepackage{makecell}
        """).lstrip())
        f.write(latex_df.to_latex(index=False, float_format="%.2f", escape=False))

    print(f"[OK] Wrote {tex_path}")
except Exception as e:
    print(f"[WARN] Could not write LaTeX table: {e}")

[OK] Wrote .\out\temp_0_extraction_choice_counts.csv

Counts and percentages by model (temperature = 0, extraction = regex_llm):
model  A  B  C  total  pct_A  pct_B  pct_C  regex_rows  llm_rows  invalid_rows
GPT4o 12 22 86    120  10.00  18.33  71.67         120         0             0
Gemma  5 12 49     66   7.58  18.18  74.24          66         0             0
Llama 15 43 62    120  12.50  35.83  51.67         120         0             0
 Qwen 13 27 80    120  10.83  22.50  66.67         120         0             0
[OK] Wrote .\out\temp_0_extraction_choice_counts_by_model.tex
